# Computational Theory: SHA-256

## Problem 1: Representing SHA-256 Data

These examples follow the [SHA-256 standard](https://doi.org/10.6028/NIST.FIPS.180-4). I used [ChatGPT](https://chatgpt.com/) to help with this section.

### 32-bit words

I use [`np.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#unsigned-integer-types) because SHA-256 uses unsigned 32-bit words. Its range is 0 to 2**32 - 1; Python's normal `int` has no fixed size.

In [1]:
import numpy as np

word = np.uint32(0x12345678)
print("Decimal:", word)
print("Binary:", f"{word:032b}")
print("Hex:", f"{word:08x}")
print("Size in bytes:", word.itemsize)

Decimal: 305419896
Binary: 00010010001101000101011001111000
Hex: 12345678
Size in bytes: 4


### Sequences of words

A NumPy array with `dtype=np.uint32` keeps the words in order, with the same type for every element.

In [2]:
words = np.array([1, 2, 3], dtype=np.uint32)
print("Words:", words)
print("Type:", words.dtype)
print("Size in bytes:", words.nbytes)

Words: [1 2 3]
Type: uint32
Size in bytes: 12


### Input messages

I use `bytes` because the input is binary data. Text is [encoded as UTF-8](https://docs.python.org/3/library/stdtypes.html#str.encode); `café` has four characters but five bytes.

In [3]:
text = "café"
message = text.encode("utf-8")
print("Message:", message)
print("Characters:", len(text))
print("Bytes:", len(message))

Message: b'caf\xc3\xa9'
Characters: 4
Bytes: 5


### 512-bit message blocks

A block is 64 bytes, split into sixteen big-endian `uint32` words for processing. This is a sample block; padding is covered in Problem 4.

In [4]:
block = bytes(range(64))
block_words = np.array(
    [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)],
    dtype=np.uint32,
)
print("Block bytes:", len(block))
print("Block words:", len(block_words))
print("First word:", f"{block_words[0]:08x}")
print("Last word:", f"{block_words[-1]:08x}")

Block bytes: 64
Block words: 16
First word: 00010203
Last word: 3c3d3e3f


### The final 256-bit hash

Eight `uint32` words make 32 bytes. I convert them to 64 lowercase hex characters for readable output. The example values only show the format.

In [5]:
hash_words = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.uint32)
digest = b"".join(int(word).to_bytes(4, "big") for word in hash_words)
hex_digest = digest.hex()
print("Digest bytes:", len(digest))
print("Hex digest:", hex_digest)
print("Hex characters:", len(hex_digest))

Digest bytes: 32
Hex digest: 0000000100000002000000030000000400000005000000060000000700000008
Hex characters: 64


### Big-endian integers

Big-endian puts the most significant byte first. I use [`int.from_bytes()` and `int.to_bytes()`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) with `"big"` to follow SHA-256's byte order.

In [6]:
value = 0x12345678
big_endian = value.to_bytes(4, "big")
little_endian = value.to_bytes(4, "little")
print("Big-endian:", big_endian.hex(" "))
print("Little-endian:", little_endian.hex(" "))
print("Read back:", hex(int.from_bytes(big_endian, "big")))

Big-endian: 12 34 56 78
Little-endian: 78 56 34 12
Read back: 0x12345678


## Problem 2: SHA-256 Bitwise Operations

The definitions follow section 4.1.2 of the [Secure Hash Standard](https://doi.org/10.6028/NIST.FIPS.180-4). I use `np.uint32` to keep operations within 32 bits. In Python, `&` means AND, `^` means XOR, and `~` means NOT.

### Ch: choose

`Ch(x, y, z) = (x & y) ^ (~x & z)`

At each bit position, `x` selects the bit from `y` when it is 1, otherwise from `z`. With `uint32`, NOT flips exactly 32 bits. The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [7]:
def Ch(x, y, z):
    """Return a uint32 choosing bits from y or z using x."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (~x & z)

In [8]:
from itertools import product

for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Ch(x, y, z)
    assert result == (y if x else z)
    assert isinstance(result, np.uint32)

assert Ch(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xCACACACA
assert Ch(0x80000000, 0xFFFFFFFF, 0) == 0x80000000
print("Ch tests passed")

Ch tests passed


### Maj: majority

`Maj(x, y, z) = (x & y) ^ (x & z) ^ (y & z)`

Each output bit is 1 when at least two input bits are 1. With three ones, XOR of the three pairwise results is also 1. The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [9]:
def Maj(x, y, z):
    """Return the bitwise majority of three words as uint32."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (x & z) ^ (y & z)

In [10]:
for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Maj(x, y, z)
    expected = 0xFFFFFFFF if (x, y, z).count(0xFFFFFFFF) >= 2 else 0
    assert result == expected
    assert isinstance(result, np.uint32)

assert Maj(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xE8E8E8E8
assert Maj(0x80000000, 0x80000000, 0) == 0x80000000
print("Maj tests passed")

Maj tests passed


## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256